In [1]:
import numpy as np 
import pandas as pd 
import joblib
from difflib import get_close_matches
from datetime import datetime, timezone
import requests
data = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/processed/full_req_data_final_final.csv")
historical_data = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/processed/historical_data_final_final.csv")
model = joblib.load("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/models/point_predicter_final.pkl")

In [2]:
API_KEY = "cb8a5495-0125-4122-9cef-e0993d41c40f"
SERIES_ID = "87c62aac-bc3c-4738-ab93-19da0690488f"
url = f"https://api.cricapi.com/v1/series_squad?apikey={API_KEY}&id={SERIES_ID}"
response = requests.get(url)
data_api_1 = response.json()
def get_next_match(series_info_json):
    match_list = series_info_json['data']['matchList']
    
    now = datetime.now(timezone.utc)
    upcoming_matches = []
    
    for match in match_list:
        # Make match_time timezone-aware (UTC)
        match_time = datetime.fromisoformat(match['dateTimeGMT']).replace(tzinfo=timezone.utc)
        
        if match_time > now and not match['matchStarted']:
            upcoming_matches.append(match)
    
    if not upcoming_matches:
        return None
    
    # Sort by actual datetime (better than string sort)
    upcoming_matches.sort(
        key=lambda x: datetime.fromisoformat(x['dateTimeGMT']).replace(tzinfo=timezone.utc)
    )
    
    next_match = upcoming_matches[0]
    
    return {
        "team1": next_match['teams'][0],
        "team2": next_match['teams'][1],
        "venue": next_match['venue'],
        "date": next_match['dateTimeGMT']
    }
def get_team_squads(data, team1, team2):
    team1_squad = []
    team2_squad = []

    for team in data['data']:
        team_name = team['teamName']

        if team_name.lower() == team1.lower():
            team1_squad = [player['name'] for player in team['players']]

        elif team_name.lower() == team2.lower():
            team2_squad = [player['name'] for player in team['players']]

    return team1_squad, team2_squad
url = f"https://api.cricapi.com/v1/series_info?apikey={API_KEY}&id={SERIES_ID}"
response = requests.get(url)
data_api_2 = response.json()
next_match = get_next_match(data_api_2)
print(next_match)

{'team1': 'Royal Challengers Bengaluru', 'team2': 'Sunrisers Hyderabad', 'venue': 'M.Chinnaswamy Stadium, Bengaluru', 'date': '2026-03-28T14:00:00'}


In [3]:
pd.set_option("display.max_columns",None)

In [4]:
team1 = next_match['team1']
team2 = next_match['team2']
venue = next_match['venue']
print(venue)
team1_squad,team2_squad = get_team_squads(data_api_1,team1,team2)
print("Team 1 Squad:", team1_squad)
print("Team 2 Squad:", team2_squad)
squad = team1_squad+team2_squad

M.Chinnaswamy Stadium, Bengaluru
Team 1 Squad: ['Bhuvneshwar Kumar', 'Virat Kohli', 'Vicky Ostwal', 'Vihaan Malhotra', 'Yash Dayal', 'Romario Shepherd', 'Jacob Bethell', 'Rajat Patidar', 'Tim David', 'Jordan Cox', 'Mangesh Yadav', 'Satvik Deswal', 'Josh Hazlewood', 'Venkatesh Iyer', 'Abhinandan Singh', 'Swapnil Singh', 'Jacob Duffy', 'Rasikh Salam Dar', 'Devdutt Padikkal', 'Jitesh Sharma', 'Krunal Pandya', 'Philip Salt', 'Kanishk Chouhan', 'Nuwan Thushara', 'Suyash Sharma']
Team 2 Squad: ['Heinrich Klaasen', 'Eshan Malinga', 'Praful Hinge', 'Nitish Kumar Reddy', 'Harshal Patel', 'Sakib Hussain', 'Smaran Ravichandran', 'Aniket Verma', 'Jack Edwards', 'Kamindu Mendis', 'Krains Fuletra', 'Shivang Kumar', 'Onkar Tukaram Tarmale', 'Abhishek Sharma', 'Shivam Mavi', 'Amit Kumar', 'Pat Cummins', 'Travis Head', 'Harsh Dubey', 'Salil Arora', 'Brydon Carse', 'Jaydev Unadkat', 'Ishan Kishan', 'Liam Livingstone', 'Zeeshan Ansari']


In [5]:
unique_player_list_from_df = historical_data['player'].unique()
unique_players = list(unique_player_list_from_df)
def map_player_name(name, unique_players):
    match = get_close_matches(name, unique_players, n=1, cutoff=0.85)
    return match[0] if match else None


mapped_players_1 = [
    m for p in team1_squad 
    if (m := map_player_name(p, unique_players)) is not None
]

mapped_players_2 = [
    m for p in team2_squad 
    if (m := map_player_name(p, unique_players)) is not None
]
all_mapped = mapped_players_1 + mapped_players_2
print("team1 :", mapped_players_1)
print("team2 :", mapped_players_2)
print(len(all_mapped))

team1 : ['Bhuvaneshwar Kumar', 'Virat Kohli', 'Yash Dayal', 'Romario Shepherd', 'Jacob Bethell', 'Rajat Patidar', 'Tim David', 'Josh Hazlewood', 'Venkatesh Iyer', 'Swapnil Singh', 'Rasikh Salam', 'Devdutt Padikkal', 'Jitesh Sharma', 'Krunal Pandya', 'Phil Salt', 'Nuwan Thushara', 'Suyash Sharma']
team2 : ['Heinrich Klaasen', 'Nithish Kumar Reddy', 'Harshal Patel', 'Aniket Verma', 'Kamindu Mendis', 'Abhishek Sharma', 'Shivam Mavi', 'Sumit Kumar', 'Pat Cummins', 'Travis Head', 'Harsh Dubey', 'Jaydev Unadkat', 'Ishan Kishan', 'Liam Livingstone', 'Zeeshan Ansari']
32


In [6]:
unique_venue_list_from_df = historical_data['venue'].unique()
unique_venues = list(unique_venue_list_from_df)
def map_venue_name(name):
    match = get_close_matches(name, unique_venues, n=1, cutoff=0.5)
    if match:
        return match[0]
    else:
        return None
mapped_venue = map_venue_name(venue)
players_df = data[
    data['player'].isin(all_mapped)
].copy()
players_df = (
    players_df
    .sort_values('player_match_number')
    .groupby('player')
    .tail(1)
)

In [7]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,venue_avg_points,venue_run_factor,batting_contribution_ratio,bowling_contribution_ratio,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,recent_form,venue_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler
8389,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,79.000000,79.000000,1.016059,1.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.800000,0,35.672040,2818.091144,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False
22417,1426270,Sumit Kumar,31.0,Narendra Modi Stadium,2024,3,35.672040,80.762144,0.0,31.000000,31.000000,1.079940,1.000000,0.000000,0.000000,0.000000,0.000000,8.0,0.888889,0,35.672040,1105.833234,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False
7270,1473505,Harsh Dubey,95.0,Arun Jaitley Stadium,2025,3,35.672040,80.762144,0.0,95.000000,95.000000,1.031603,0.000000,1.000000,0.000000,0.000000,0.000000,11.0,0.000000,0,35.672040,3388.843781,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True
15270,1473499,Kamindu Mendis,43.0,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2025,5,27.666667,80.762144,0.0,43.000000,43.000000,1.040207,1.000000,0.000000,42.821334,0.000000,0.000000,5.0,0.375000,0,30.868816,1327.359085,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False
14333,1473507,Nuwan Thushara,35.0,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2025,8,38.666667,0.000000,1.6,54.333333,34.000000,1.040207,0.000000,1.000000,41.383226,0.000000,0.000000,11.0,0.000000,0,40.687204,1383.364935,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True
25124,1473499,Zeeshan Ansari,-2.0,Bharat Ratna Shri Atal Bihari Vajpayee Ekana C...,2025,10,9.000000,0.000000,0.4,-2.000000,-2.000000,1.040207,-0.000000,1.000000,27.459060,0.000000,0.000000,11.0,0.000000,0,12.807204,-25.614408,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True
2477,1473505,Aniket Verma,15.0,Arun Jaitley Stadium,2025,12,19.333333,153.492063,0.0,15.000000,15.000000,1.031603,1.000000,0.000000,29.522872,1.138101,-14.266667,5.0,0.833333,0,21.080000,316.200000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,Tr

In [8]:
players_df['venue'] = mapped_venue
players_df = players_df.drop(columns = ['venue_avg_points','venue_run_factor'])
venue_data = data[data['venue'] == mapped_venue][['player','venue_avg_points','venue_run_factor']].drop_duplicates()
players_df = players_df.merge(venue_data,on = ['player'],how = 'left')
players_df['venue_avg_points'] = players_df['venue_avg_points'].fillna(venue_data['venue_avg_points'].mean())
players_df['venue_run_factor'] = players_df['venue_run_factor'].fillna(venue_data['venue_run_factor'].mode()[0])
players_df = players_df.drop(columns = ['venue_form'])
players_df['venue_form'] = players_df['venue_avg_points'] * players_df['recent_form']

In [9]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,batting_contribution_ratio,bowling_contribution_ratio,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form
0,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,79.000000,1.000000,0.000000,0.000000,0.000000,0.000000,1.0,0.800000,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,79.000000,1.016059,2818.091144
1,1426270,Sumit Kumar,31.0,M Chinnaswamy Stadium,2024,3,35.672040,80.762144,0.0,31.000000,1.000000,0.000000,0.000000,0.000000,0.000000,8.0,0.888889,0,35.672040,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,32.584401,1.016059,1162.352052
2,1473505,Harsh Dubey,95.0,M Chinnaswamy Stadium,2025,3,35.672040,80.762144,0.0,95.000000,0.000000,1.000000,0.000000,0.000000,0.000000,11.0,0.000000,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,32.584401,1.016059,1162.352052
3,1473499,Kamindu Mendis,43.0,M Chinnaswamy Stadium,2025,5,27.666667,80.762144,0.0,43.000000,1.000000,0.000000,42.821334,0.000000,0.000000,5.0,0.375000,0,30.868816,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,32.584401,1.016059,1005.841879
4,1473507,Nuwan Thushara,35.0,M Chinnaswamy Stadium,2025,8,38.666667,0.000000,1.6,54.333333,0.000000,1.000000,41.383226,0.000000,0.000000,11.0,0.000000,0,40.687204,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,32.584401,1.016059,1325.768173
5,1473499,Zeeshan Ansari,-2.0,M Chinnaswamy Stadium,2025,10,9.000000,0.000000,0.4,-2.000000,-0.000000,1.000000,27.459060,0.000000,0.000000,11.0,0.000000,0,12.807204,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,32.584401,1.016059,417.315071
6,1473505,Aniket Verma,15.0,M Chinnaswamy Stadium,2025,12,19.333333,153.492063,0.0,15.000000,1.000000,0.000000,29.522872,1.138101,-14.266667,5.0,0.833333,0,21.080000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,32.584401,1.016059,686.879175
7,1473451,Rasikh Salam,-4.0,M Chinnaswamy Stadium,2025,13,29.333333,92.666667,0.8,44.500000,-0.000

In [10]:
ratio_info = historical_data[['player','player_match_number','last3_battingcontri','last3_bowlingcontri','last3_boundarypercentage']]
ratio_df = (
    ratio_info
    .sort_values('player_match_number')
    .groupby('player')
    .tail(1)
)
ratio_df = ratio_df.drop(columns = ['player_match_number'])
players_df = players_df.merge(ratio_df,on = ['player'],how = 'left')
players_df = players_df.drop(columns = ['batting_contribution_ratio','bowling_contribution_ratio','boundary_percentage'])
players_df = players_df.rename(columns = {"last3_battingcontri":"batting_contribution_ratio","last3_bowlingcontri":"bowling_contribution_ratio",'last3_boundarypercentage':'boundary_percentage'})

In [11]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,last10_std_points,player_consistency_index,form_momentum,bat_pos,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form,batting_contribution_ratio,bowling_contribution_ratio,boundary_percentage
0,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,79.000000,0.000000,0.000000,0.000000,1.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,79.000000,1.016059,2818.091144,1.000000,0.000000,0.816667
1,1426270,Sumit Kumar,31.0,M Chinnaswamy Stadium,2024,3,35.672040,80.762144,0.0,31.000000,0.000000,0.000000,0.000000,8.0,0,35.672040,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,32.584401,1.016059,1162.352052,1.000000,0.000000,0.582011
2,1473505,Harsh Dubey,95.0,M Chinnaswamy Stadium,2025,3,35.672040,80.762144,0.0,95.000000,0.000000,0.000000,0.000000,11.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,False,True,32.584401,1.016059,1162.352052,0.000000,1.000000,0.000000
3,1473499,Kamindu Mendis,43.0,M Chinnaswamy Stadium,2025,5,27.666667,80.762144,0.0,43.000000,42.821334,0.000000,0.000000,5.0,0,30.868816,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,32.584401,1.016059,1005.841879,0.855263,0.144737,0.250000
4,1473507,Nuwan Thushara,35.0,M Chinnaswamy Stadium,2025,8,38.666667,0.000000,1.6,54.333333,41.383226,0.000000,0.000000,11.0,0,40.687204,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,32.584401,1.016059,1325.768173,0.000000,1.000000,0.000000
5,1473499,Zeeshan Ansari,-2.0,M Chinnaswamy Stadium,2025,10,9.000000,0.000000,0.4,-2.000000,27.459060,0.000000,0.000000,11.0,0,12.807204,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,32.584401,1.016059,417.315071,-0.000000,1.000000,0.000000
6,1473505,Aniket Verma,15.0,M Chinnaswamy Stadium,2025,12,19.333333,153.492063,0.0,15.000000,29.522872,1.138101,-14.266667,5.0,0,21.080000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,False,True,False,False,True,False,32.584401,1.016059,686.879175,1.000000,0.000000,0.826496
7,1473451,Rasikh Salam,-4.0,M Chinnaswamy Stadium,2025,13,29.333333,92.666667,0.8,44.500000,36.644

In [12]:
team_cols = [col for col in players_df.columns if col.startswith('team_')]
opponent_cols = [col for col in players_df.columns if col.startswith('opponent_')]
team_cols = team_cols[1:] # Removing team_won_toss from list
opponent_cols = opponent_cols[1:] # Removing opponent_avg_points from list
players_df[team_cols] = False
players_df[opponent_cols] = False
if f"team_{team1}" in team_cols:
    players_df.loc[players_df['player'].isin(mapped_players_1), f"team_{team1}"] = True

if f"team_{team2}" in team_cols:
    players_df.loc[players_df['player'].isin(mapped_players_2), f"team_{team2}"] = True
if f"opponent_{team2}" in opponent_cols:
    players_df.loc[players_df['player'].isin(mapped_players_1), f"opponent_{team2}"] = True

if f"opponent_{team1}" in opponent_cols:
    players_df.loc[players_df['player'].isin(mapped_players_2), f"opponent_{team1}"] = True

In [13]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,last10_std_points,player_consistency_index,form_momentum,bat_pos,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form,batting_contribution_ratio,bowling_contribution_ratio,boundary_percentage
0,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,79.000000,0.000000,0.000000,0.000000,1.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,79.000000,1.016059,2818.091144,1.000000,0.000000,0.816667
1,1426270,Sumit Kumar,31.0,M Chinnaswamy Stadium,2024,3,35.672040,80.762144,0.0,31.000000,0.000000,0.000000,0.000000,8.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,32.584401,1.016059,1162.352052,1.000000,0.000000,0.582011
2,1473505,Harsh Dubey,95.0,M Chinnaswamy Stadium,2025,3,35.672040,80.762144,0.0,95.000000,0.000000,0.000000,0.000000,11.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,32.584401,1.016059,1162.352052,0.000000,1.000000,0.000000
3,1473499,Kamindu Mendis,43.0,M Chinnaswamy Stadium,2025,5,27.666667,80.762144,0.0,43.000000,42.821334,0.000000,0.000000,5.0,0,30.868816,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,32.584401,1.016059,1005.841879,0.855263,0.144737,0.250000
4,1473507,Nuwan Thushara,35.0,M Chinnaswamy Stadium,2025,8,38.666667,0.000000,1.6,54.333333,41.383226,0.000000,0.000000,11.0,0,40.687204,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,32.584401,1.016059,1325.768173,0.000000,1.000000,0.000000
5,1473499,Zeeshan Ansari,-2.0,M Chinnaswamy Stadium,2025,10,9.000000,0.000000,0.4,-2.000000,27.459060,0.000000,0.000000,11.0,0,12.807204,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,32.584401,1.016059,417.315071,-0.000000,1.000000,0.000000
6,1473505,Aniket Verma,15.0,M Chinnaswamy Stadium,2025,12,19.333333,153.492063,0.0,15.000000,29.522872,1.138101,-14.266667,5.0,0,21.080000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,32.584401,1.016059,686.879175,1.000000,0.000000,0.826496
7,1473451,Rasikh Salam,-4.0,M Chinnaswamy Stadium,2025,13,29.333333,92.666667,0.8,44.500000,36.6448

In [14]:
opponent_avg_dict = historical_data.groupby(['player','opponent'])['fantasy_points'].mean().to_dict()
player_avg_dict = (
    historical_data
    .groupby('player')['fantasy_points']
    .mean()
    .to_dict()
)
opponent_cols = [c for c in players_df.columns if c.startswith('opponent_')]
opponent_cols = opponent_cols[1:]
players_df['opponent_temp'] = (
    players_df[opponent_cols]
    .idxmax(axis=1)
    .str.replace('opponent_', '')
)
all_opponents = historical_data['opponent'].unique()
encoded_opponents = [c.replace('opponent_', '') for c in opponent_cols]

dropped_opponent = list(set(all_opponents) - set(encoded_opponents))[0]

mask = players_df[opponent_cols].sum(axis=1) == 0
players_df.loc[mask, 'opponent_temp'] = dropped_opponent
def get_opponent_avg(row):
    key = (row['player'], row['opponent_temp'])
    
    if key in opponent_avg_dict:
        return opponent_avg_dict[key]
    else:
        return player_avg_dict.get(row['player'], 0)
    
players_df = players_df.drop(columns = ['opponent_avg_points'])
players_df['opponent_avg_points'] = players_df.apply(get_opponent_avg, axis=1)
players_df = players_df.drop(columns = ['opponent_temp'])

In [15]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,last10_std_points,player_consistency_index,form_momentum,bat_pos,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form,batting_contribution_ratio,bowling_contribution_ratio,boundary_percentage,opponent_avg_points
0,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,1.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,79.000000,1.016059,2818.091144,1.000000,0.000000,0.816667,51.000000
1,1426270,Sumit Kumar,31.0,M Chinnaswamy Stadium,2024,3,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,8.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,32.584401,1.016059,1162.352052,1.000000,0.000000,0.582011,14.000000
2,1473505,Harsh Dubey,95.0,M Chinnaswamy Stadium,2025,3,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,11.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,32.584401,1.016059,1162.352052,0.000000,1.000000,0.000000,23.000000
3,1473499,Kamindu Mendis,43.0,M Chinnaswamy Stadium,2025,5,27.666667,80.762144,0.0,42.821334,0.000000,0.000000,5.0,0,30.868816,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,False,32.584401,1.016059,1005.841879,0.855263,0.144737,0.250000,37.000000
4,1473507,Nuwan Thushara,35.0,M Chinnaswamy Stadium,2025,8,38.666667,0.000000,1.6,41.383226,0.000000,0.000000,11.0,0,40.687204,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,True,32.584401,1.016059,1325.768173,0.000000,1.000000,0.000000,-2.000000
5,1473499,Zeeshan Ansari,-2.0,M Chinnaswamy Stadium,2025,10,9.000000,0.000000,0.4,27.459060,0.000000,0.000000,11.0,0,12.807204,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,True,32.584401,1.016059,417.315071,-0.000000,1.000000,0.000000,17.800000
6,1473505,Aniket Verma,15.0,M Chinnaswamy Stadium,2025,12,19.333333,153.492063,0.0,29.522872,1.138101,-14.266667,5.0,0,21.080000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,32.584401,1.016059,686.879175,1.000000,0.000000,0.826496,41.000000
7,1473451,Rasikh Salam,-4.0,M Chinnaswamy Stadium,2025,13,29.333333,92.666667,0.8,36.644842,0.82412

In [16]:
pitch_info = historical_data[historical_data['venue'] == mapped_venue]['pitch_type'].mode()[0]
pitch_cols = [
    'pitch_type_batting_friendly',
    'pitch_type_pace_friendly',
    'pitch_type_spin_friendly'
]
players_df[pitch_cols] = False
if pitch_info == 'batting_friendly':
    players_df['pitch_type_batting_friendly'] = True
elif pitch_info == 'pace_friendly':
    players_df['pitch_type_pace_friendly'] = True
elif pitch_info == 'spin_friendly':
    players_df['pitch_type_spin_friendly'] = True

In [17]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,last10_std_points,player_consistency_index,form_momentum,bat_pos,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form,batting_contribution_ratio,bowling_contribution_ratio,boundary_percentage,opponent_avg_points
0,1473489,Jacob Bethell,79.0,M Chinnaswamy Stadium,2025,2,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,1.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,79.000000,1.016059,2818.091144,1.000000,0.000000,0.816667,51.000000
1,1426270,Sumit Kumar,31.0,M Chinnaswamy Stadium,2024,3,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,8.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,32.584401,1.016059,1162.352052,1.000000,0.000000,0.582011,14.000000
2,1473505,Harsh Dubey,95.0,M Chinnaswamy Stadium,2025,3,35.672040,80.762144,0.0,0.000000,0.000000,0.000000,11.0,0,35.672040,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,32.584401,1.016059,1162.352052,0.000000,1.000000,0.000000,23.000000
3,1473499,Kamindu Mendis,43.0,M Chinnaswamy Stadium,2025,5,27.666667,80.762144,0.0,42.821334,0.000000,0.000000,5.0,0,30.868816,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,32.584401,1.016059,1005.841879,0.855263,0.144737,0.250000,37.000000
4,1473507,Nuwan Thushara,35.0,M Chinnaswamy Stadium,2025,8,38.666667,0.000000,1.6,41.383226,0.000000,0.000000,11.0,0,40.687204,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,False,True,32.584401,1.016059,1325.768173,0.000000,1.000000,0.000000,-2.000000
5,1473499,Zeeshan Ansari,-2.0,M Chinnaswamy Stadium,2025,10,9.000000,0.000000,0.4,27.459060,0.000000,0.000000,11.0,0,12.807204,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,True,32.584401,1.016059,417.315071,-0.000000,1.000000,0.000000,17.800000
6,1473505,Aniket Verma,15.0,M Chinnaswamy Stadium,2025,12,19.333333,153.492063,0.0,29.522872,1.138101,-14.266667,5.0,0,21.080000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,32.584401,1.016059,686.879175,1.000000,0.000000,0.826496,41.000000
7,1473451,Rasikh Salam,-4.0,M Chinnaswamy Stadium,2025,13,29.333333,92.666667,0.8,36.644842,0.824127,-0

In [18]:
X_pred = players_df.drop(columns=['fantasy_points','player','venue','player_match_number','match_id','season','player_role_batsman','player_role_bowler'])
players_df['predicted_points'] = model.predict(X_pred)
team_map = {}
for player in mapped_players_1:
    team_map[player] = team1
for player in mapped_players_2:
    team_map[player] = team2
players_df['team'] = players_df['player'].map(team_map)
player_stats = historical_data.groupby('player').agg({
    'batting_contribution_ratio': 'mean',
    'bowling_contribution_ratio': 'mean'
}).reset_index()
def assign_role(row):
    bat = row['batting_contribution_ratio']
    bowl = row['bowling_contribution_ratio']
    
    # All-rounder: contributes in BOTH
    if bat > 0.15 and bowl > 0.15:
        return 'AR'
    
    # Pure batsman
    elif bat >= bowl:
        return 'BAT'
    
    # Pure bowler
    else:
        return 'BOWL'
player_stats['role'] = player_stats.apply(assign_role, axis=1)
role_map = dict(zip(player_stats['player'], player_stats['role']))
players_df['role'] = players_df['player'].map(role_map).fillna('BAT')
players_df = players_df.sort_values(by='predicted_points', ascending=False).reset_index(drop=True)
def get_role(row):
    if row['player_role_wicketkeeper']:
        return 'WK'
    return row['role'].upper()

players_df['final_role'] = players_df.apply(get_role, axis=1)
def build_valid_team(players_df):
    team = []
    team_count = {}
    
    role_constraints = {
        'WK': (1, 4),
        'BAT': (3, 6),
        'AR': (1, 4),
        'BOWL': (3, 6)
    }

    role_count = {r: 0 for r in role_constraints}

    # ✅ Step 1: Fill minimum requirements
    for role, (min_req, max_req) in role_constraints.items():
        candidates = players_df[
            (players_df['final_role'] == role) &
            (~players_df['player'].isin(team))
        ]
        
        for _, row in candidates.iterrows():
            if role_count[role] >= min_req:
                break
                
            team_name = row['team']
            
            if team_count.get(team_name, 0) >= 7:
                continue
            
            team.append(row['player'])
            role_count[role] += 1
            team_count[team_name] = team_count.get(team_name, 0) + 1

    # ✅ Step 2: Fill remaining slots (best players)
    for _, row in players_df.iterrows():
        if len(team) >= 11:
            break
            
        player = row['player']
        role = row['final_role']
        team_name = row['team']
        
        if player in team:
            continue
        
        # Team constraint
        if team_count.get(team_name, 0) >= 7:
            continue
        
        # Role max constraint
        if role_count[role] >= role_constraints[role][1]:
            continue
        
        team.append(player)
        role_count[role] += 1
        team_count[team_name] = team_count.get(team_name, 0) + 1

    return team, role_count, team_count
def choose_captains(team, players_df):
    selected = players_df[players_df['player'].isin(team)]
    selected = selected.sort_values(by='predicted_points', ascending=False)
    
    captain = selected.iloc[0]['player']
    vice_captain = selected.iloc[1]['player']
    
    return captain, vice_captain
team, role_count, team_count = build_valid_team(players_df)
captain, vice_captain = choose_captains(team, players_df)
print("🏏 Final Team:")
print(team)

print("\n📊 Role Distribution:")
print(role_count)

print("\n🏢 Team Distribution:")
print(team_count)

print("\n👑 Captain:", captain)
print("🤝 Vice-Captain:", vice_captain)   # Will add for 💰 Budget constraint, 2. Improve team selection (BIGGEST GAIN)
#👉 Optimal Dream11 solver using ILP (guaranteed best team) later'''


🏏 Final Team:
['Ishan Kishan', 'Abhishek Sharma', 'Liam Livingstone', 'Phil Salt', 'Kamindu Mendis', 'Zeeshan Ansari', 'Josh Hazlewood', 'Shivam Mavi', 'Jitesh Sharma', 'Heinrich Klaasen', 'Tim David']

📊 Role Distribution:
{'WK': 3, 'BAT': 4, 'AR': 1, 'BOWL': 3}

🏢 Team Distribution:
{'Sunrisers Hyderabad': 7, 'Royal Challengers Bengaluru': 4}

👑 Captain: Ishan Kishan
🤝 Vice-Captain: Jitesh Sharma


In [19]:
players_df

,match_id,player,fantasy_points,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,last10_std_points,player_consistency_index,form_momentum,bat_pos,player_role_wicketkeeper,recent_form,team_Deccan Chargers,team_Delhi Capitals,team_Gujarat Lions,team_Gujarat Titans,team_Kochi Tuskers Kerala,team_Kolkata Knight Riders,team_Lucknow Super Giants,team_Mumbai Indians,team_Pune Warriors,team_Punjab Kings,team_Rajasthan Royals,team_Rising Pune Supergiants,team_Royal Challengers Bengaluru,team_Sunrisers Hyderabad,opponent_Deccan Chargers,opponent_Delhi Capitals,opponent_Gujarat Lions,opponent_Gujarat Titans,opponent_Kochi Tuskers Kerala,opponent_Kolkata Knight Riders,opponent_Lucknow Super Giants,opponent_Mumbai Indians,opponent_Pune Warriors,opponent_Punjab Kings,opponent_Rajasthan Royals,opponent_Rising Pune Supergiants,opponent_Royal Challengers Bengaluru,opponent_Sunrisers Hyderabad,pitch_type_batting_friendly,pitch_type_pace_friendly,pitch_type_spin_friendly,player_role_batsman,player_role_bowler,venue_avg_points,venue_run_factor,venue_form,batting_contribution_ratio,bowling_contribution_ratio,boundary_percentage,opponent_avg_points,predicted_points,team,role,final_role
0,1473505,Ishan Kishan,37.0,M Chinnaswamy Stadium,2025,114,70.000000,110.343137,0.0,42.744850,0.741610,38.300000,2.0,1,61.610000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,13.333333,1.016059,821.466667,1.000000,0.000000,0.687119,48.133333,23.879980,Sunrisers Hyderabad,BAT,WK
1,1473511,Jitesh Sharma,44.0,M Chinnaswamy Stadium,2025,47,58.666667,170.125000,0.0,36.118478,1.165608,16.566667,5.0,1,54.950000,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,33.500000,1.016059,1840.825000,1.000000,0.000000,0.766667,33.166667,20.565156,Royal Challengers Bengaluru,BAT,WK
2,1473505,Heinrich Klaasen,152.0,M Chinnaswamy Stadium,2025,45,48.333333,145.822788,0.0,22.803509,2.192645,-1.666667,5.0,1,49.420000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,59.500000,1.016059,2940.490000,1.000000,0.000000,0.694124,80.200000,18.159444,Sunrisers Hyderabad,BAT,WK
3,1473505,Abhishek Sharma,52.0,M Chinnaswamy Stadium,2025,77,50.000000,137.597561,0.0,60.734029,0.968156,-8.800000,1.0,0,53.640000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,48.000000,1.016059,2574.720000,1.045455,-0.045455,0.879570,38.700000,4.728231,Sunrisers Hyderabad,BAT,BAT
4,1473511,Liam Livingstone,41.0,M Chinnaswamy Stadium,2025,48,1.000000,26.666667,0.0,33.213786,0.764743,-24.400000,4.0,0,4.400000,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,28.800000,1.016059,126.720000,1.000000,0.000000,0.493333,26.000000,4.117440,Sunrisers Hyderabad,BAT,BAT
5,1473511,Phil Salt,28.0,M Chinnaswamy Stadium,2025,34,77.333333,142.335791,0.0,37.452340,1.246918,30.633333,1.0,0,67.330000,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,True,False,False,True,False,37.800000,1.016059,2545.074000,1.000000,0.000000,0.808333,70.250000,3.749038,Royal Challengers Bengaluru,BAT,BAT
6,1473505,Travis Head,108.0,M Chinnaswamy Stadium,2025,37,24.333333,102.750000,0.0,31.774203,1.057462,-9.266667,1.0,0,25.040000,False,False,False,False,False,False,False,False,False,False,False,

In [20]:
historical_data

,match_id,player,fantasy_points,team,opponent,venue,season,player_match_number,last3_avg_points,rolling_strike_rate,rolling_wickets,opponent_avg_points,pitch_type,venue_avg_points,venue_run_factor,batting_contribution_ratio,bowling_contribution_ratio,player_role,last10_std_points,player_consistency_index,form_momentum,bat_pos,boundary_percentage,player_role_wicketkeeper,recent_form,venue_form,last3_battingcontri,last3_bowlingcontri,last3_boundarypercentage
0,548341,Ashish Reddy,58.0,Deccan Chargers,Pune Warriors,Subrata Roy Sahara Stadium,2012,1,35.672040,80.762144,0.0,39.250000,balanced,58.00,0.879923,0.000000,1.000000,bowler,0.000000,0.0,0.0,7.0,0.0,0,35.672040,2068.978308,0.000000,1.000000,0.0
1,548346,Ashish Reddy,41.0,Deccan Chargers,Mumbai Indians,Wankhede Stadium,2012,2,35.672040,80.762144,0.0,30.000000,batting_friendly,41.00,1.035196,0.292683,0.707317,allrounder,0.000000,0.0,0.0,7.0,0.6,0,35.672040,1462.553632,0.146341,0.853659,0.3
2,548348,Ashish Reddy,23.0,Deccan Chargers,Pune Warriors,Barabati Stadium,2012,3,35.672040,80.762144,0.0,39.250000,balanced,23.00,1.027630,0.000000,1.000000,bowler,0.000000,0.0,0.0,7.0,0.0,0,35.672040,820.456915,0.097561,0.902439,0.2
3,548352,Ashish Reddy,28.0,Deccan Chargers,Chennai Super Kings,MA Chidambaram Stadium,2012,4,40.666667,80.762144,0.0,47.333333,spin_friendly,45.00,0.984847,0.107143,0.892857,bowler,17.502381,0.0,0.0,7.0,0.0,0,38.668816,1740.096716,0.133275,0.866725,0.2
4,548356,Ashish Reddy,33.0,Deccan Chargers,Royal Challengers Bengaluru,M Chinnaswamy Stadium,2012,5,30.666667,80.762144,0.0,38.714286,batting_friendly,31.75,1.016059,0.000000,1.000000,bowler,15.631165,0.0,0.0,7.0,0.0,0,32.668816,1037.234905,0.035714,0.964286,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25120,1473478,Zeeshan Ansari,25.0,Sunrisers Hyderabad,Mumbai Indians,Rajiv Gandhi International Stadium,2025,6,3.333333,0.000000,0.8,18.500000,balanced,6.75,0.994802,0.000000,1.000000,bowler,36.004166,0.0,0.0,11.0,0.0,0,13.247204,89.418627,0.000000,1.000000,0.0
25121,1473480,Zeeshan Ansari,0.0,Sunrisers Hyderabad,Chennai Super Kings,MA Chidambaram Stadium,2025,7,11.666667,0.000000,0.4,0.000000,spin_friendly,0.00,0.984847,0.000000,0.000000,allrounder,32.204037,0.0,0.0,11.0,0.0,0,14.647204,0.000000,0.000000,0.666667,0.0
25122,1473488,Zeeshan Ansari,23.0,Sunrisers Hyderabad,Gujarat Titans,Narendra Modi Stadium,2025,8,12.333333,0.000000,0.2,11.500000,balanced,23.00,1.079940,0.000000,1.000000,bowler,30.937725,0.0,0.0,11.0,0.0,0,13.067204,300.545692,0.000000,0.666667,0.0
25123,1473492,Zeeshan Ansari,4.0,Sunrisers Hyderabad,Delhi Capitals,Rajiv Gandhi International Stadium,2025,9,16.000000,0.000000,0.4,44.500000,balanced,6.75,0.994802,-0.000000,1.000000,bowler,28.645618,0.0,0.0,11.0,0.0,0,16.647204,112.368627,-0.000000,0.666667,0.0


In [22]:
X_pred.columns

Index(['last3_avg_points', 'rolling_strike_rate', 'rolling_wickets',
       'last10_std_points', 'player_consistency_index', 'form_momentum',
       'bat_pos', 'player_role_wicketkeeper', 'recent_form',
       'team_Deccan Chargers', 'team_Delhi Capitals', 'team_Gujarat Lions',
       'team_Gujarat Titans', 'team_Kochi Tuskers Kerala',
       'team_Kolkata Knight Riders', 'team_Lucknow Super Giants',
       'team_Mumbai Indians', 'team_Pune Warriors', 'team_Punjab Kings',
       'team_Rajasthan Royals', 'team_Rising Pune Supergiants',
       'team_Royal Challengers Bengaluru', 'team_Sunrisers Hyderabad',
       'opponent_Deccan Chargers', 'opponent_Delhi Capitals',
       'opponent_Gujarat Lions', 'opponent_Gujarat Titans',
       'opponent_Kochi Tuskers Kerala', 'opponent_Kolkata Knight Riders',
       'opponent_Lucknow Super Giants', 'opponent_Mumbai Indians',
       'opponent_Pune Warriors', 'opponent_Punjab Kings',
       'opponent_Rajasthan Royals', 'opponent_Rising Pune Supergian

In [24]:
data3 = pd.read_csv("/Users/nidhishgupta/Desktop/Dream11_Fantasy Team_Predictor/data/half_processed/half_prepared_data.csv")

In [26]:
data3.columns.to_list()

['match_id',
 'player',
 'runs',
 'balls',
 'fours',
 'sixes',
 'strike_rate',
 'runs_conceded',
 'balls_bowled',
 'wickets',
 'overs',
 'economy',
 'fielding_points',
 'batting_points',
 'batting_bonus',
 'sr_bonus',
 'wicket_bonus',
 'eco_bonus',
 'lbw_bowled_bonus',
 'bowling_points',
 'maiden_points',
 'fantasy_points',
 'team',
 'opponent',
 'venue',
 'city',
 'toss_winner',
 'toss_decision',
 'stage',
 'season',
 'date',
 'player_match_number',
 'last3_avg_points',
 'last5_avg_points',
 'last10_avg_points',
 'rolling_strike_rate',
 'rolling_wickets',
 'opponent_avg_points',
 'pitch_type',
 'venue_avg_points',
 'venue_run_factor',
 'batting_contribution_ratio',
 'bowling_contribution_ratio',
 'player_role',
 'last10_std_points',
 'player_consistency_index',
 'form_momentum',
 'bat_pos',
 'boundary_percentage',
 'bat_pos_per_match']

In [27]:
historical_data.columns

Index(['match_id', 'player', 'fantasy_points', 'team', 'opponent', 'venue',
       'season', 'player_match_number', 'last3_avg_points',
       'rolling_strike_rate', 'rolling_wickets', 'opponent_avg_points',
       'pitch_type', 'venue_avg_points', 'venue_run_factor',
       'batting_contribution_ratio', 'bowling_contribution_ratio',
       'player_role', 'last10_std_points', 'player_consistency_index',
       'form_momentum', 'bat_pos', 'boundary_percentage',
       'player_role_wicketkeeper', 'recent_form', 'venue_form',
       'last3_battingcontri', 'last3_bowlingcontri',
       'last3_boundarypercentage'],
      dtype='object')